In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os, shutil, glob, zipfile, re

DRIVE_DIR   = "/content/drive/MyDrive/Corneal_v2"            # <- your upload folder
CSV_NAME    = "Final_Features_MultiLabel_updated.csv"
IMG_NAME    = "Disease_Cell_Images"                           # zip OR folder with this name
LOCAL_DIR   = "/content/data_v2"
IMAGE_DIR   = os.path.join(LOCAL_DIR, "images")
RESULTS_DIR = "/content/drive/MyDrive/CNN_results_v2"         # NEW results folder
os.makedirs(LOCAL_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SEV_NAME    = "Severity_Labels_PerDisease_updated.xlsx"
CSV_PATH = os.path.join(DRIVE_DIR, CSV_NAME)
SEV_FILE = os.path.join(DRIVE_DIR, SEV_NAME)
assert os.path.exists(SEV_FILE), f"Severity Excel not found: {SEV_FILE}"
assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"

if not os.path.isdir(IMAGE_DIR):
    zip_path, folder_path = os.path.join(DRIVE_DIR, IMG_NAME + ".zip"), os.path.join(DRIVE_DIR, IMG_NAME)
    if os.path.exists(zip_path):
        print("Unzipping", zip_path); zipfile.ZipFile(zip_path).extractall(IMAGE_DIR)
    elif os.path.isdir(folder_path):
        print("Copying folder", folder_path); shutil.copytree(folder_path, IMAGE_DIR)
    else:
        raise FileNotFoundError(f"Put {IMG_NAME}.zip or the {IMG_NAME}/ folder inside {DRIVE_DIR}")

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
ALL_IMGS = [p for p in glob.glob(os.path.join(IMAGE_DIR, "**", "*"), recursive=True)
            if p.lower().endswith(IMG_EXT) and "__MACOSX" not in p]
print("Images found (all sub-folders):", len(ALL_IMGS))
print("Example file names:", [os.path.relpath(p, IMAGE_DIR) for p in ALL_IMGS[:8]])

Unzipping /content/drive/MyDrive/Corneal_v2/Disease_Cell_Images.zip
Images found (all sub-folders): 1543
Example file names: ['Disease_Cell_Images/NEB-2023-08-15288 R_SP.jpg', 'Disease_Cell_Images/NEB-2022-04-13822 R_SP.jpg', 'Disease_Cell_Images/NEB-2024-06-16136 R_SP.jpg', 'Disease_Cell_Images/NEB-2023-04-14947 R_SP.jpg', 'Disease_Cell_Images/NEB-2025-09-17464 L_SP.jpg', 'Disease_Cell_Images/NEB-2023-03-14825 R_SP.jpg', 'Disease_Cell_Images/NEB-2023-03-14907 R_SP.jpg', 'Disease_Cell_Images/NEB-2025-12-17726 L_SP.jpg']


In [5]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
print("CSV rows:", len(df))

def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())
def short_key(s):
    m = re.search(r"(\d{4,6})[\s_\-]*([LR])(?![A-Z])", str(s).upper())
    return f"{m.group(1)}{m.group(2)}" if m else None

full_map, short_map, dup = {}, {}, []
for p in ALL_IMGS:
    stem = os.path.splitext(os.path.basename(p))[0]
    k = norm(stem)
    if k in full_map: dup.append(stem)
    full_map.setdefault(k, p)
    sk = short_key(stem)
    if sk: short_map.setdefault(sk, []).append(p)

# pass 1: exact (normalised) name match
df["Img_Path"] = df["Tissue_ID"].map(lambda t: full_map.get(norm(t)))
# pass 2: tissue-number + eye fallback, only for images not already taken and only if unambiguous
taken = set(df["Img_Path"].dropna())
sk_rows = df[df["Img_Path"].isna()]["Tissue_ID"].map(short_key)
for i, k in sk_rows.items():
    cands = [p for p in short_map.get(k or "", []) if p not in taken]
    if len(cands) == 1 and (sk_rows == k).sum() == 1:
        df.at[i, "Img_Path"] = cands[0]; taken.add(cands[0])
unmatched = df[df["Img_Path"].isna()]
print(f"Matched: {df['Img_Path'].notna().sum()} / {len(df)}   | unmatched rows: {len(unmatched)}")
if dup: print("WARNING duplicate image names (first kept):", dup[:5])
if len(unmatched):
    print("Unmatched Tissue_IDs (first 15):", unmatched["Tissue_ID"].head(15).tolist())
used = set(df["Img_Path"].dropna())
print("Images not used by any CSV row:", len(set(ALL_IMGS) - used))
n_dup = df["Img_Path"].dropna().duplicated().sum()
print("Image used by >1 row:", n_dup)
assert n_dup == 0, "Two rows point to the same image - check file names"

# Stop if matching is clearly wrong
assert df["Img_Path"].notna().mean() > 0.9, "Less than 90% of rows matched an image - fix img matching before training!"
df = df[df["Img_Path"].notna()].reset_index(drop=True)
unmatched.to_csv(os.path.join(RESULTS_DIR, "unmatched_rows.csv"), index=False)

CSV rows: 1548
Matched: 1527 / 1548   | unmatched rows: 21
WARNING duplicate image names (first kept): ['NEB-2026-03-18084L_SP', 'NEB-2021-11 -13399 R_SP']
Unmatched Tissue_IDs (first 15): ['NEB-2020-09-12672 R', 'NEB-2020-09-12683 L', 'NEB-2020-10-12726 R', 'NEB-2020-10-12733 L', 'NEB-2022-10-14315 L', 'NEB-2022-10-14349 L', 'NEB-2023-02-14710 R', 'NEB-2023-04-14996 R', 'NEB-2023-08-15300 R', 'NEB-2024-06-16116 L', 'NEB-2024-06-16117 R', 'NEB-2024-06-16130 L', 'NEB-2024-08-16256 L', 'NEB-2024-08-16296 L', 'NEB-2025-09-17506 R']
Images not used by any CSV row: 16
Image used by >1 row: 0


In [6]:
import json, time, copy, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, f1_score, confusion_matrix
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NONE - turn on T4 GPU!")

device: cuda | GPU: Tesla T4


Configure the model and severity labels

In [7]:
BACKBONE = "densenet121"
PRETRAINED = True
IMG_W, IMG_H = 384, 288
EPOCHS, WARMUP_EPOCHS, PATIENCE = 30, 2, 7
BATCH_SIZE = 16
LR_HEAD, LR_BACKBONE, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
SEED = 42
N_FOLDS = 5
DIS_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]
SEV_HEADS = {"FECDgroup": ["FECD", "PolyPleo", "LowCD"], "Folds": ["Folds"], "Guttata": ["Guttata"]}   # severity outputs
SEV_COLS = list(SEV_HEADS)
FEATURES = ["ECD_CD_cells_per_mm2", "ECD_Deficit", "SD", "CV", "HEX_percent", "Donor_Age"]
RUN_DIR = os.path.join(RESULTS_DIR, f"severity_{BACKBONE}")
os.makedirs(RUN_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# --- shared donor-grouped CV folds (same file as the fusion notebooks) ---
fold_file = os.path.join(RESULTS_DIR, "cv_folds.csv")
if os.path.exists(fold_file):
    df = df.drop(columns=["CV_Fold"], errors="ignore").merge(pd.read_csv(fold_file), on="Tissue_ID", how="left")
    print("Loaded existing folds from", fold_file)
else:
    df["CV_Fold"] = -1
    tr_idx = df.index[df["Split"] == "train"]
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(sgkf.split(tr_idx, df.loc[tr_idx, "Disease_Label"], df.loc[tr_idx, "Donor_Group"])):
        df.loc[tr_idx[va], "CV_Fold"] = k
    df[["Tissue_ID", "CV_Fold"]].to_csv(fold_file, index=False)
    print("Created new donor-grouped folds ->", fold_file)

# --- severity labels ---
sev = pd.read_excel(SEV_FILE, sheet_name="Severity_Labels", keep_default_na=False, na_values=[""])
sev = sev[["Tissue_ID"] + [f"{h}_Sev" for h in SEV_COLS] + (["Overall_Sev"] if "Overall_Sev" not in [f"{h}_Sev" for h in SEV_COLS] else [])]
df = df.merge(sev, on="Tissue_ID", how="left")
print("Rows without severity info:", df[f"{SEV_COLS[0]}_Sev"].isna().sum())

for h in SEV_COLS:
    s = df[f"{h}_Sev"].astype(str)
    df[f"SevY_{h}"] = s.isin(["Moderate", "Severe"]).astype(float)
    df[f"SevM_{h}"] = s.isin(["Mild", "Moderate", "Severe"]).astype(float)
    # sanity: severity only where the disease is present
    has_dis = df[SEV_HEADS[h]].max(axis=1) == 1
    print(f"{h}: severity known but disease absent = {int(((df[f'SevM_{h}'] == 1) & ~has_dis).sum())}")
df["Label_Origin"] = np.where(df["Tissue_ID"].str.contains("NEB-2023"), "Model-assisted (2023)", "Original")

train_df = df[df["Split"] == "train"].reset_index(drop=True)
test_df  = df[df["Split"] == "test"].reset_index(drop=True)
assert not set(train_df["Donor_Group"]) & set(test_df["Donor_Group"]), "Donor leakage between train and test!"
assert train_df["CV_Fold"].between(0, N_FOLDS - 1).all(), "Some train rows have no fold - delete cv_folds.csv and re-run"
summary = pd.DataFrame({f"{h} {p}": [int(d[f"SevM_{h}"].sum() - d[f"SevY_{h}"].sum()) if p == "Mild" else int(d[f"SevY_{h}"].sum())
                                      for d in (train_df, test_df)] for h in SEV_COLS for p in ("Mild", "Mod+Sev")},
                       index=["Train", "Test"])
print(f"{BACKBONE} | train {len(train_df)} | test {len(test_df)} | folds {N_FOLDS}")
summary

Loaded existing folds from /content/drive/MyDrive/CNN_results_v2/cv_folds.csv
Rows without severity info: 0
FECDgroup: severity known but disease absent = 0
Folds: severity known but disease absent = 0
Guttata: severity known but disease absent = 0
densenet121 | train 1222 | test 305 | folds 5


,FECDgroup Mild,FECDgroup Mod+Sev,Folds Mild,Folds Mod+Sev,Guttata Mild,Guttata Mod+Sev
Train,201,185,390,225,112,40
Test,50,48,96,57,26,11


Prepares and augments the images, handles missing numerical values, and scales the biomarkers.

In [8]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
T = torchvision.transforms
TF_TRAIN = T.Compose([T.ToPILImage(),
                      T.RandomResizedCrop((IMG_H, IMG_W), scale=(0.7, 1.0), ratio=(1.25, 1.42)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])
TF_EVAL = T.Compose([T.ToPILImage(), T.Resize((IMG_H, IMG_W)),
                     T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])

def fit_tab_scaler(frame):
    imp = SimpleImputer(strategy="median").fit(frame[FEATURES])
    sc = StandardScaler().fit(imp.transform(frame[FEATURES]))
    return lambda f: sc.transform(imp.transform(f[FEATURES])).astype(np.float32)

class SPDataset(Dataset):
    def __init__(self, frame, tab, train):
        self.paths = list(frame["Img_Path"])
        self.Yd = frame[DIS_COLS].values.astype(np.float32)
        self.Ys = frame[[f"SevY_{h}" for h in SEV_COLS]].values.astype(np.float32)
        self.Ms = frame[[f"SevM_{h}" for h in SEV_COLS]].values.astype(np.float32)
        self.tab = torch.tensor(tab, dtype=torch.float32)
        self.tf = TF_TRAIN if train else TF_EVAL
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = self.tf(cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE))
        return x, self.tab[i], torch.tensor(self.Yd[i]), torch.tensor(self.Ys[i]), torch.tensor(self.Ms[i])

class FusionMultiTask(nn.Module):
    """Image (DenseNet121) + 6 numbers -> shared features -> 5 disease outputs + severity outputs."""
    def __init__(self):
        super().__init__()
        m = models.densenet121(weights="DEFAULT" if PRETRAINED else None)
        dim = m.classifier.in_features; m.classifier = nn.Identity()
        self.backbone, self.cam_layer = m, m.features
        self.tab_mlp = nn.Sequential(nn.Linear(len(FEATURES), 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 32), nn.ReLU())
        self.shared = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim + 32, 128), nn.ReLU(), nn.Dropout(0.3))
        self.disease_head = nn.Linear(128, len(DIS_COLS))
        self.severity_head = nn.Linear(128, len(SEV_COLS))
    def forward(self, x, tab):
        z = self.shared(torch.cat([self.backbone(x), self.tab_mlp(tab)], 1))
        return self.disease_head(z), self.severity_head(z)

def pos_weight(Y, M=None):
    M = np.ones_like(Y) if M is None else M
    pos = (Y * M).sum(0); neg = M.sum(0) - pos
    return torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=DEVICE)

def make_loader(frame, scale, train):
    return DataLoader(SPDataset(frame, scale(frame), train), batch_size=BATCH_SIZE if train else 32,
                      shuffle=train, drop_last=train, num_workers=2, pin_memory=True)

@torch.no_grad()
def predict(model, dl):
    model.eval(); PD, PS = [], []
    for x, tab, *_ in dl:
        x, tab = x.to(DEVICE), tab.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            d1, s1 = model(x, tab); d2, s2 = model(torch.flip(x, dims=[3]), tab)
        PD.append(((torch.sigmoid(d1) + torch.sigmoid(d2)) / 2).float().cpu().numpy())
        PS.append(((torch.sigmoid(s1) + torch.sigmoid(s2)) / 2).float().cpu().numpy())
    return np.concatenate(PD), np.concatenate(PS)

def safe_auc(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan

def val_score(frame, PD, PS):
    d = np.nanmean([safe_auc(frame[c].values, PD[:, j]) for j, c in enumerate(DIS_COLS)])
    s = []
    for j, h in enumerate(SEV_COLS):
        m = frame[f"SevM_{h}"].values == 1
        s.append(safe_auc(frame.loc[m, f"SevY_{h}"].values, PS[m, j]))
    return 0.5 * d + 0.5 * np.nanmean(s), d, s

def train_one(tr_frame, va_frame=None, epochs=EPOCHS, log=True):
    set_seed()
    scale = fit_tab_scaler(tr_frame)
    tr_dl = make_loader(tr_frame, scale, True)
    va_dl = None if va_frame is None else make_loader(va_frame, scale, False)
    model = FusionMultiTask().to(DEVICE)
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    crit_d = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[DIS_COLS].values))
    crit_s = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[[f"SevY_{h}" for h in SEV_COLS]].values,
                                                        tr_frame[[f"SevM_{h}" for h in SEV_COLS]].values), reduction="none")
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": head, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - WARMUP_EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_sc, best_ep, wait = None, -1, epochs, 0
    for ep in range(1, epochs + 1):
        warm = ep <= WARMUP_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad = not warm
        model.train(); t0, losses = time.time(), []
        for x, tab, yd, ys, ms in tr_dl:
            x, tab, yd, ys, ms = [t.to(DEVICE) for t in (x, tab, yd, ys, ms)]
            opt.zero_grad()
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                ld, ls = model(x, tab)
                loss_d = crit_d(ld, yd)
                loss_s = (crit_s(ls.float(), ys) * ms).sum() / ms.sum().clamp(min=1)   # masked: only known severities
                loss = loss_d + loss_s
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
        if not warm: sched.step()
        msg = f"  ep {ep:2d} loss {np.mean(losses):.4f}"
        if va_dl is not None:
            sc, d, s = val_score(va_frame, *predict(model, va_dl))
            msg += f" | val disease AUC {d:.3f} | val severity AUC " + " / ".join(f"{v:.3f}" for v in s)
            if sc > best_sc: best_sc, best_ep, best, wait = sc, ep, copy.deepcopy(model.state_dict()), 0
            elif not warm: wait += 1
        if log: print(msg + (" (warm-up)" if warm else "") + f" ({time.time() - t0:.0f}s)")
        if va_dl is not None and wait >= PATIENCE: break
    if best is not None: model.load_state_dict(best)
    for p in model.parameters(): p.requires_grad = True
    return model, scale, best_ep

print(f"OK - multi-task model ready (5 disease + {len(SEV_COLS)} severity outputs: {SEV_COLS})")

OK - multi-task model ready (5 disease + 3 severity outputs: ['FECDgroup', 'Folds', 'Guttata'])


Trains the multi-task model using five folds, rotating the validation fold each time

In [9]:
oof_path, info_path = os.path.join(RUN_DIR, "oof_predictions.csv"), os.path.join(RUN_DIR, "fold_info.json")
pcols = [f"P_{c}" for c in DIS_COLS] + [f"PSev_{h}" for h in SEV_COLS]
oof = pd.read_csv(oof_path) if os.path.exists(oof_path) else train_df[["Tissue_ID", "CV_Fold"]].assign(**{c: np.nan for c in pcols})
assert list(oof.columns[2:]) == pcols, "Old oof file has different outputs - delete RUN_DIR/oof_predictions.csv and fold_info.json"
folds = json.load(open(info_path)) if os.path.exists(info_path) else {}

for k in range(N_FOLDS):
    if str(k) in folds:
        print(f"Fold {k} already done - skipping"); continue
    tr_f, va_f = train_df[train_df["CV_Fold"] != k], train_df[train_df["CV_Fold"] == k]
    print(f"\n===== Severity multi-task | fold {k} | train {len(tr_f)} / val {len(va_f)} =====")
    t0 = time.time()
    model, scale, best_ep = train_one(tr_f, va_f)
    PD, PS = predict(model, make_loader(va_f, scale, False))
    oof.loc[oof["Tissue_ID"].isin(va_f["Tissue_ID"]), pcols] = np.hstack([PD, PS])
    oof.to_csv(oof_path, index=False)
    sc, d, s = val_score(va_f, PD, PS)
    folds[str(k)] = {"best_epoch": int(best_ep), "disease_auc": float(d), "severity_auc": [float(v) for v in s],
                     "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(folds, open(info_path, "w"), indent=2)
    print(f"Fold {k}: disease AUC {d:.3f} | severity AUC {[round(v, 3) for v in s]} | best epoch {best_ep} - SAVED to Drive")
    del model; torch.cuda.empty_cache()


===== Severity multi-task | fold 0 | train 992 / val 230 =====
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 164MB/s]


  ep  1 loss 1.8963 | val disease AUC 0.720 | val severity AUC 0.719 / 0.682 / 0.443 (warm-up) (26s)
  ep  2 loss 1.7306 | val disease AUC 0.789 | val severity AUC 0.571 / 0.688 / 0.362 (warm-up) (22s)
  ep  3 loss 1.5396 | val disease AUC 0.804 | val severity AUC 0.639 / 0.701 / 0.328 (26s)
  ep  4 loss 1.4800 | val disease AUC 0.829 | val severity AUC 0.695 / 0.746 / 0.287 (25s)
  ep  5 loss 1.4580 | val disease AUC 0.854 | val severity AUC 0.692 / 0.734 / 0.276 (25s)
  ep  6 loss 1.3941 | val disease AUC 0.848 | val severity AUC 0.720 / 0.718 / 0.276 (25s)
  ep  7 loss 1.3787 | val disease AUC 0.862 | val severity AUC 0.749 / 0.775 / 0.293 (25s)
  ep  8 loss 1.3219 | val disease AUC 0.857 | val severity AUC 0.759 / 0.748 / 0.362 (25s)
  ep  9 loss 1.2861 | val disease AUC 0.872 | val severity AUC 0.792 / 0.817 / 0.489 (25s)
  ep 10 loss 1.2839 | val disease AUC 0.874 | val severity AUC 0.805 / 0.826 / 0.615 (25s)
  ep 11 loss 1.2905 | val disease AUC 0.873 | val severity AUC 0.732 /

Compares severity prediction using image-plus-numerical features against a numbers-only Random Forest baseline.

In [10]:
o = train_df.merge(oof[["Tissue_ID"] + pcols], on="Tissue_ID")
rf = lambda: RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced_subsample",
                                    random_state=SEED, n_jobs=-1)
rows, THR = [], {}
for j, h in enumerate(SEV_COLS):
    m = o[f"SevM_{h}"] == 1
    y, p = o.loc[m, f"SevY_{h}"].values, o.loc[m, f"PSev_{h}"].values
    num_p = np.zeros(m.sum())                                  # numbers-only baseline on the SAME folds
    sub = o[m].reset_index(drop=True)
    for k in range(N_FOLDS):
        tr, va = sub["CV_Fold"] != k, sub["CV_Fold"] == k
        if va.sum() == 0 or sub.loc[tr, f"SevY_{h}"].nunique() < 2: continue
        sc_ = fit_tab_scaler(sub[tr]); mdl = rf().fit(sc_(sub[tr]), sub.loc[tr, f"SevY_{h}"])
        num_p[va.values] = mdl.predict_proba(sc_(sub[va]))[:, 1]
    fpr, tpr, thr = roc_curve(y, p); b = np.argmax(tpr - fpr); THR[h] = float(np.clip(thr[b], 0.01, 0.99))
    rows.append({"Severity output": h, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                 "Numbers only AUC": safe_auc(y, num_p), "Image + numbers AUC": safe_auc(y, p),
                 "Sensitivity": tpr[b], "Specificity": 1 - fpr[b], "Threshold": THR[h]})
cv_sev = pd.DataFrame(rows).set_index("Severity output")
cv_dis = pd.Series({c: safe_auc(o[c].values, o[f"P_{c}"].values) for c in DIS_COLS}, name="CV disease AUC")
DTHR = {}
for c in DIS_COLS:
    fpr, tpr, thr = roc_curve(o[c], o[f"P_{c}"]); DTHR[c] = float(np.clip(thr[np.argmax(tpr - fpr)], 0.01, 0.99))
cv_sev.to_csv(os.path.join(RUN_DIR, "cv_severity.csv"))
print("TRAIN CV - severity (Mild vs Moderate/Severe), only corneas that HAVE the disease")
display(cv_sev.round(3))
print("TRAIN CV - disease detection of the same multi-task model")
display(cv_dis.round(3).to_frame().T)

TRAIN CV - severity (Mild vs Moderate/Severe), only corneas that HAVE the disease


,n (Mild / Mod+Sev),Numbers only AUC,Image + numbers AUC,Sensitivity,Specificity,Threshold
Severity output,,,,,,
FECDgroup,201 / 185,0.731,0.783,0.649,0.801,0.570
Folds,390 / 225,0.700,0.807,0.711,0.769,0.491
Guttata,112 / 40,0.638,0.745,0.700,0.705,0.434


TRAIN CV - disease detection of the same multi-task model


,FECD,Guttata,Folds,PolyPleo,LowCD
CV disease AUC,0.87,0.815,0.766,0.865,0.985


Uses the median best epoch from cross-validation to determine the final training duration.

In [11]:
ep = max(int(np.median([v["best_epoch"] for v in folds.values()])), WARMUP_EPOCHS + 1)
print(f"Final multi-task model: {ep} epochs on all {len(train_df)} train images")
final_model, final_scale, _ = train_one(train_df, None, epochs=ep)
PD_TE, PS_TE = predict(final_model, make_loader(test_df, final_scale, False))
torch.save(final_model.state_dict(), os.path.join(RUN_DIR, "final_multitask.pt"))
t = test_df.copy()
for j, c in enumerate(DIS_COLS): t[f"P_{c}"] = PD_TE[:, j]
for j, h in enumerate(SEV_COLS): t[f"PSev_{h}"] = PS_TE[:, j]
print("OK - test predictions ready")

Final multi-task model: 10 epochs on all 1222 train images
  ep  1 loss 1.9215 (warm-up) (21s)
  ep  2 loss 1.7465 (warm-up) (21s)
  ep  3 loss 1.5544 (25s)
  ep  4 loss 1.5083 (25s)
  ep  5 loss 1.4572 (25s)
  ep  6 loss 1.4076 (25s)
  ep  7 loss 1.3586 (25s)
  ep  8 loss 1.3010 (25s)
  ep  9 loss 1.2884 (25s)
  ep 10 loss 1.2435 (25s)
OK - test predictions ready


Evaluates severity predictions against the actual test labels and compares them with the numbers-only baseline.

In [12]:
rng = np.random.default_rng(SEED)
def auc_ci(y, p, n=1000):
    s = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i): s.append(roc_auc_score(y[i], p[i]))
    return (np.percentile(s, 2.5), np.percentile(s, 97.5)) if s else (np.nan, np.nan)

rows = []
for h in SEV_COLS:
    trm = train_df[f"SevM_{h}"] == 1
    sc_ = fit_tab_scaler(train_df[trm]); mdl = rf().fit(sc_(train_df[trm]), train_df.loc[trm, f"SevY_{h}"])
    m = t[f"SevM_{h}"] == 1
    y, p = t.loc[m, f"SevY_{h}"].values, t.loc[m, f"PSev_{h}"].values
    pn = mdl.predict_proba(sc_(t[m]))[:, 1]
    yhat = (p >= THR[h]).astype(int)
    lo, hi = auc_ci(y, p)
    rows.append({"Severity output": h, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                 "Numbers only AUC": safe_auc(y, pn), "Image + numbers AUC": safe_auc(y, p),
                 "95% CI": f"[{lo:.2f}-{hi:.2f}]", "Sensitivity": recall_score(y, yhat, zero_division=0),
                 "Specificity": recall_score(y, yhat, pos_label=0, zero_division=0)})
test_sev = pd.DataFrame(rows).set_index("Severity output")
test_sev.to_csv(os.path.join(RUN_DIR, "test_severity.csv"))
print("HELD-OUT TEST - severity (Mild vs Moderate/Severe), corneas that truly have the disease")
display(test_sev.round(3))

test_dis = pd.Series({c: safe_auc(t[c].values, t[f"P_{c}"].values) for c in DIS_COLS}, name="Test disease AUC")
test_dis.to_csv(os.path.join(RUN_DIR, "test_disease_auc.csv"))
print("HELD-OUT TEST - disease detection of the multi-task model"); display(test_dis.round(3).to_frame().T)

rows = []
for h in SEV_COLS:
    for org in ["Original", "Model-assisted (2023)"]:
        m = (t[f"SevM_{h}"] == 1) & (t["Label_Origin"] == org)
        y, p = t.loc[m, f"SevY_{h}"].values, t.loc[m, f"PSev_{h}"].values
        rows.append({"Severity output": h, "Label origin": org, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                     "AUC": safe_auc(y, p)})
print("Test severity AUC by label origin (NaN = not enough cases of both classes)")
display(pd.DataFrame(rows).round(3))

HELD-OUT TEST - severity (Mild vs Moderate/Severe), corneas that truly have the disease


,n (Mild / Mod+Sev),Numbers only AUC,Image + numbers AUC,95% CI,Sensitivity,Specificity
Severity output,,,,,,
FECDgroup,50 / 48,0.697,0.847,[0.77-0.92],0.771,0.820
Folds,96 / 57,0.630,0.849,[0.78-0.91],0.737,0.781
Guttata,26 / 11,0.678,0.647,[0.45-0.83],0.545,0.692


HELD-OUT TEST - disease detection of the multi-task model


,FECD,Guttata,Folds,PolyPleo,LowCD
Test disease AUC,0.905,0.822,0.803,0.859,0.984


Test severity AUC by label origin (NaN = not enough cases of both classes)


,Severity output,Label origin,n (Mild / Mod+Sev),AUC
0,FECDgroup,Original,32 / 40,0.842
1,FECDgroup,Model-assisted (2023),18 / 8,0.847
2,Folds,Original,62 / 49,0.828
3,Folds,Model-assisted (2023),34 / 8,0.868
4,Guttata,Original,18 / 10,0.556
5,Guttata,Model-assisted (2023),8 / 1,0.875


Combines disease detection and severity predictions into an overall cornea report.

In [13]:
ORD = {"None": 0, "Mild": 1, "Mod/Sev": 2}
HEAD_NAME = {"FECDgroup": "FECD group", "Folds": "Folds", "Guttata": "Guttata"}
def cornea_report(r):
    found = [c for c in DIS_COLS if r[f"P_{c}"] >= DTHR[c]]
    sevs = []
    for h in SEV_COLS:
        if any(c in found for c in SEV_HEADS[h]):
            sevs.append((HEAD_NAME.get(h, h), "Mod/Sev" if r[f"PSev_{h}"] >= THR[h] else "Mild"))
    if "Guttata" in found and "Guttata" not in SEV_HEADS:
        sevs.append(("Guttata", "Mild"))          # only if Guttata has no severity output
    overall = max([s for _, s in sevs], key=lambda s: ORD[s]) if sevs else "None"
    return pd.Series({"Predicted diseases": " + ".join(found) if found else "Normal",
                      "Predicted severity": ", ".join(f"{d}: {s}" for d, s in sevs) if sevs else "-",
                      "Predicted overall": overall})
rep = pd.concat([t[["Tissue_ID", "Combination", "Overall_Sev"]], t.apply(cornea_report, axis=1)], axis=1)
rep["True overall"] = rep["Overall_Sev"].astype(str).replace({"Moderate": "Mod/Sev", "Severe": "Mod/Sev", "?": "Mod/Sev", "nan": "None"})
labels = ["None", "Mild", "Mod/Sev"]
cm = pd.DataFrame(confusion_matrix(rep["True overall"], rep["Predicted overall"], labels=labels),
                  index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
print("End-to-end overall cornea severity (test): true vs predicted")
display(cm)
print(f"Overall severity exact agreement: {(rep['True overall'] == rep['Predicted overall']).mean():.1%}")
print("\nExample reports (test):")
display(rep.drop(columns="Overall_Sev").sample(min(12, len(rep)), random_state=1).reset_index(drop=True))

End-to-end overall cornea severity (test): true vs predicted


,Pred None,Pred Mild,Pred Mod/Sev
True None,66,28,7
True Mild,15,84,34
True Mod/Sev,0,20,51


Overall severity exact agreement: 65.9%

Example reports (test):


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
0,NEB-2020-04-12255 L,Folds + LowCD,FECD + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev",Mod/Sev,Mod/Sev
1,NEB-2025-10-17594 L,Normal,Normal,-,None,None
2,NEB-2022-06-13962 R,Folds + PolyPleo + LowCD,FECD + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev",Mod/Sev,Mod/Sev
3,NEB-2025-09-17516 L,Normal,Normal,-,None,None
4,NEB-2020-01-11908 R,Folds,FECD + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev",Mod/Sev,Mod/Sev
5,NEB-2023-03-14819 L,Folds + PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mild
6,NEB-2020-01-11957 R,PolyPleo,FECD + Guttata + PolyPleo + LowCD,"FECD group: Mild, Guttata: Mild",Mild,Mild
7,NEB-2023-10-15438 R,Guttata,Guttata,Guttata: Mild,Mild,Mild
8,NEB-2023-04-14994 L,FECD + Guttata + Folds + LowCD,FECD + Guttata + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev, Guttata: ...",Mod/Sev,Mod/Sev
9,NEB-2020-02-12069 L,Folds,Folds,Folds: Mod/Sev,Mod/Sev,Mod/Sev


In [14]:
rep.drop(columns="Overall_Sev").to_csv(os.path.join(RUN_DIR, "test_cornea_reports.csv"), index=False)
t[["Tissue_ID", "Label_Origin"] + DIS_COLS + [f"SevY_{h}" for h in SEV_COLS] + [f"SevM_{h}" for h in SEV_COLS]
  + [f"P_{c}" for c in DIS_COLS] + [f"PSev_{h}" for h in SEV_COLS]].to_csv(os.path.join(RUN_DIR, "test_predictions_severity.csv"), index=False)
json.dump({"backbone": BACKBONE, "dataset": CSV_NAME, "severity_file": SEV_NAME,
           "n_train": len(train_df), "n_test": len(test_df),
           "severity_heads": SEV_HEADS, "severity_thresholds": THR, "disease_thresholds": DTHR,
           "cv_severity": cv_sev.round(4).to_dict(orient="index"), "test_severity": test_sev.round(4).to_dict(orient="index"),
           "cv_disease_auc": cv_dis.round(4).to_dict(), "test_disease_auc": test_dis.round(4).to_dict(),
           "overall_confusion": cm.to_dict(), "epochs_final": ep, "folds": folds},
          open(os.path.join(RUN_DIR, "summary_severity.json"), "w"), indent=2, default=str)
print("Saved to", RUN_DIR); print(os.listdir(RUN_DIR))

Saved to /content/drive/MyDrive/CNN_results_v2/severity_densenet121
['oof_predictions.csv', 'fold_info.json', 'cv_severity.csv', 'final_multitask.pt', 'test_severity.csv', 'test_disease_auc.csv', 'test_cornea_reports.csv', 'test_predictions_severity.csv', 'summary_severity.json']
